# dots.tts 语音合成面板（小红书 · Colab 版）

一键启动**公网面板**：输入文字 → 选音色（预设 / 上传克隆 / 音色库）→ 选语言 → 出语音。

**面板功能：**
- ✅ 界面与语言选项**全中文**
- ✅ **音色预设**：内置 15 个多语言音色（中文 / 英语 / 日韩法德 / 情绪参考），点「试听」可预览
- ✅ **参考音频转写**：上传人声 → 自动识别文字 → 可手动更正
- ✅ **音色库**：把上传的声音保存到 Drive，以后直接选
- ✅ **音色相似度** + 高级设置（音色种子 / 生成质量 / 引导强度 / 文本规范化）

**环境 + 模型都缓存到你的 Google Drive**：
- 首次装好后，**断连重开不用再重装环境**（约 1-2 分钟秒恢复）
- 模型复制到本地 SSD 加载，**不用每次从 Drive 慢读 5GB**


## 第 0 步：确认 GPU（菜单操作，不是代码）

**运行时 → 更改运行时类型 → 硬件加速器选 GPU**，然后跑下面这格确认。


In [ ]:
!nvidia-smi


## 使用流程（先看清楚，能省不少时间）

**首次使用**：从上到下依次跑「第 0 步 → 第 4 步」（约 5-8 分钟，会自动安装 + 缓存）。

**以后每次 / 断连重开**：**只跑最后的「🚀 一键启动」这一格**（约 2-4 分钟，免重装、模型秒加载）。

> 分步的第 1-4 步和「一键启动」做的事完全一样，只是拆开方便你看懂 / 排错。日常只用「一键启动」这一格即可。


## 第 1 步：挂载 Google Drive + 定义路径

把环境备份、模型、音色库都放在你的 Drive 上，这样断连后不丢。


In [ ]:
# ---- 第 1 步：挂载 Google Drive + 定义路径 ----
import os, subprocess, time, re, shutil, sys

CACHE = "/content/drive/MyDrive/dots_cache"        # Drive 持久化目录（环境 + 模型 + 音色库）
PY = "/content/py311/bin/python"                    # 本地 Python 环境
ENV_TARBALL = os.path.join(CACHE, "py311.tar.gz")   # 环境备份包（首次装好后存到 Drive）
LOCAL_HF = "/content/dots_hf_cache"                 # 本地 SSD 模型缓存（加载快）
PANEL_PY = "/content/panel.py"

try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    DRIVE_OK = True
    print("✅ Drive 已挂载，持久化目录：", CACHE, flush=True)
except Exception as e:
    DRIVE_OK = False
    print("⚠️ Drive 未挂载（断连后需重装环境 + 重下模型）：", e, flush=True)


## 第 2 步：准备环境（首次安装 / 之后秒恢复）

- **第一次**：安装全部依赖（约 3-5 分钟），然后**打包缓存到 Drive**。
- **之后每次**：直接从 Drive 解包恢复（约 1-2 分钟），**不再重装**。
- **自愈**：每次都会校验 gradio / torch / dots.tts 等依赖是否齐全，**缺哪个自动补哪个**（不会出现「环境在但缺包」的情况）。


In [ ]:
# ---- 第 2 步：准备环境（首次安装并缓存到 Drive，之后秒恢复；缺依赖会自动补齐）----
import os, subprocess

_REQ = ["torch", "gradio", "dots_tts", "faster_whisper", "soundfile", "huggingface_hub"]

def _py_runs(p):
    try:
        return subprocess.run([p, "--version"], capture_output=True, text=True, timeout=60).returncode == 0
    except Exception:
        return False

def _deps_ok(p):
    _code = "import importlib.util as u, sys; sys.exit(0 if all(u.find_spec(m) for m in %r) else 1)" % (_REQ,)
    try:
        return subprocess.run([p, "-c", _code], capture_output=True, text=True, timeout=120).returncode == 0
    except Exception:
        return False

def _install_deps():
    subprocess.run("pip install -q uv", shell=True, check=True)
    UV = "uv pip install --python /content/py311/bin/python"
    subprocess.run(UV + " torch==2.11.0 torchaudio==2.11.0", shell=True, check=True)
    subprocess.run(UV + " dots.tts huggingface_hub soundfile 'gradio>=6.17,<7' faster-whisper", shell=True, check=True)

# 1) 确保 python 解释器存在（恢复 / 新建）
if not _py_runs(PY):
    if DRIVE_OK and os.path.exists(ENV_TARBALL):
        print("🔄 从 Drive 恢复环境（约 1-2 分钟，免重装）...", flush=True)
        subprocess.run(["tar", "xzf", ENV_TARBALL, "-C", "/"], check=True)
    if not _py_runs(PY):
        print("🔄 首次安装环境（约 3-5 分钟）...", flush=True)
        subprocess.run("python3 -m venv /content/py311", shell=True, check=True)

# 2) 检查依赖是否齐全，缺哪个补哪个（uv 幂等，已装的秒过）
if not _deps_ok(PY):
    print("🔄 检测到依赖缺失，正在补齐（已装的会自动跳过）...", flush=True)
    _install_deps()

if not _deps_ok(PY):
    _r = subprocess.run([PY, "-c", "import gradio"], capture_output=True, text=True)
    print("❌ 依赖仍缺失：", _r.stderr[-800:], flush=True)
    raise SystemExit("依赖安装失败，请检查上方报错后重跑本格")

# 3) 环境齐了，首次打包缓存到 Drive（之后断连免重装）
if DRIVE_OK and not os.path.exists(ENV_TARBALL):
    print("📦 缓存环境到 Drive（首次稍慢，约 2-4 分钟）...", flush=True)
    subprocess.run(["tar", "czf", ENV_TARBALL, "-C", "/", "content/py311"], check=True)
    print("✅ 环境已缓存：", ENV_TARBALL, flush=True)

print("✅ 环境就绪（含 gradio / torch / dots.tts / faster-whisper）", flush=True)


## 第 3 步：准备模型 + 写面板代码

把 Drive 上的模型缓存**复制到本地 SSD**（加载比直接从 Drive 读快数倍），并写入面板源码。

> 首次运行时 Drive 还没有模型，会直接下载到 Drive。


In [ ]:
# ---- 第 3 步：准备模型（复制到本地 SSD，加载快）+ 写面板代码 ----
import os, subprocess

panel_code = r"""import os, json, shutil, time
import gradio as gr
import soundfile as sf
import torch
from dots_tts.runtime import DotsTtsRuntime
from dots_tts.utils.util import seed_everything

# ---------- 加载模型 ----------
print("HF_HOME =", os.environ.get("HF_HOME"), flush=True)
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
PRECISION = "bfloat16" if cap[0] >= 8 else "float16"
print("加载模型...", flush=True)
runtime = DotsTtsRuntime.from_pretrained("dots-studio/dots.tts-soar", precision=PRECISION, optimize=False)
print("模型加载完成", flush=True)

# ---------- 目录 ----------
CACHE = os.environ.get("HF_HOME") or ("/content/drive/MyDrive/dots_cache" if os.path.isdir("/content/drive/MyDrive") else None)
LIB_DIR = os.path.join(CACHE, "voice_library") if CACHE else "/content/voice_library"
PRESET_DIR = "/content/presets"
os.makedirs(LIB_DIR, exist_ok=True)
os.makedirs(PRESET_DIR, exist_ok=True)
LIB_JSON = os.path.join(LIB_DIR, "voices.json")

# ---------- 内置音色预设（运行时从 GitHub 仓库下载参考音频，避免内嵌 base64 拖慢代码页） ----------
# 每项：(key, 标签, 文件名, 参考文本)。参考文本必须与音频实际内容一致。
PRESET_DEFS = [
    # 中文
    ("tingting", "婷婷（温柔女声）", "tingting.wav", "大家好，我是你的专属语音助手。今天天气很不错，我们一起来聊一聊最近发生的趣事吧。"),
    ("eddy", "埃迪（沉稳男声）", "eddy.wav", "各位听众朋友，大家好。欢迎收听今天的节目，希望你能喜欢我的声音，也祝你度过愉快的一天。"),
    ("meijia", "美佳（甜美女声）", "meijia.wav", "你好呀，很高兴认识你。今天想跟你分享一些有趣的事情，希望你听了以后会开心一点。"),
    ("sandy", "桑迪（知性女声）", "sandy.wav", "大家好，我是你的语音助手。无论是工作还是生活，我都愿意随时为你提供帮助和建议。"),
    # 英语
    ("samantha", "Samantha（美式女声）", "samantha.wav", "Hello, I'm your friendly voice assistant. It's a pleasure to meet you, and I'm here to help with whatever you need today."),
    ("daniel", "Daniel（英式男声）", "daniel.wav", "Good day to you, and welcome. I hope you find my voice clear, natural, and pleasant to listen to."),
    ("karen", "Karen（澳洲女声）", "karen.wav", "G'day, I'm your voice assistant. Let's get started and make the most of today, together."),
    ("moira", "Moira（爱尔兰女声）", "moira.wav", "Hello there, lovely to meet you. I'll be guiding you through today, so just relax and enjoy the conversation."),
    # 其他语言
    ("kyoko", "Kyoko（日语女声）", "kyoko.wav", "こんにちは、あなたの音声アシスタントです。今日もよろしくお願いします。"),
    ("yuna", "Yuna（韩语女声）", "yuna.wav", "안녕하세요, 저는 당신의 음성 비서입니다. 오늘도 좋은 하루 보내세요."),
    ("thomas", "Thomas（法语男声）", "thomas.wav", "Bonjour, je suis votre assistant vocal. C'est un plaisir de vous accompagner aujourd'hui."),
    ("anna", "Anna（德语女声）", "anna.wav", "Hallo, ich bin deine Sprachassistentin. Schön, dich heute begleiten zu dürfen."),
    # 情绪参考（英文，用来把情绪/语气转移到合成结果；合成中文会带英文口音，适合做情绪参考）
    ("goodnews", "开心·欢快（情绪参考）", "goodnews.wav", "Great news! Everything went perfectly today!"),
    ("badnews", "悲伤·低沉（情绪参考）", "badnews.wav", "I'm afraid I have some difficult news to share with you."),
    ("whisper", "悄悄话·耳语（情绪参考）", "whisper.wav", "Psst, come a little closer. I have a secret to tell you, but just between us, quietly."),
]
PRESET_BASE = "https://raw.githubusercontent.com/Evan78s/dots-tts-panel/main/presets"

PRESET_LABELS = {}
PRESET_TEXTS = {}
for _key, _label, _file, _text in PRESET_DEFS:
    PRESET_LABELS[_key] = _label
    PRESET_TEXTS[_key] = _text

def _ensure_presets():
    import urllib.request
    for _key, _label, _file, _text in PRESET_DEFS:
        _path = os.path.join(PRESET_DIR, _file)
        if os.path.exists(_path) and os.path.getsize(_path) > 1000:
            continue
        try:
            urllib.request.urlretrieve(PRESET_BASE + "/" + _file, _path)
            print("下载预设音色：%s" % _label, flush=True)
        except Exception as _e:
            print("⚠️ 预设音色「%s」下载失败（仍可上传参考音频使用）：%s" % (_label, _e), flush=True)

_ensure_presets()
print("内置音色预设：", list(PRESET_LABELS.values()), flush=True)

PRESET_CHOICES = [("默认音色（不克隆）", "")] + [(lbl, key) for key, lbl in PRESET_LABELS.items()]

# ---------- 语言（全部中文显示） ----------
LANG_CHOICES = [
    ("自动检测", "auto_detect"),
    ("中文（普通话）", "ZH"),
    ("英语", "EN"),
    ("粤语", "Cantonese"),
    ("日语", "JA"),
    ("韩语", "KO"),
    ("法语", "FR"),
    ("德语", "DE"),
    ("西班牙语", "ES"),
    ("俄语", "RU"),
    ("阿拉伯语", "AR"),
    ("印地语", "HI"),
    ("葡萄牙语", "PT"),
    ("意大利语", "IT"),
    ("泰语", "TH"),
    ("越南语", "VI"),
    ("印尼语", "ID"),
    ("捷克语", "CS"),
    ("荷兰语", "NL"),
    ("芬兰语", "FI"),
    ("希腊语", "EL"),
    ("波兰语", "PL"),
    ("罗马尼亚语", "RO"),
    ("土耳其语", "TR"),
    ("乌克兰语", "UK"),
    ("口音：北京官话", "口音:北京官话"),
    ("口音：东北话", "口音:东北话"),
    ("口音：四川话", "口音:四川话"),
    ("口音：闽南话", "口音:闽南话"),
    ("口音：吴语", "口音:吴语"),
]

# ---------- 音色库（持久化到 Drive） ----------
def load_library():
    if os.path.exists(LIB_JSON):
        try:
            return json.load(open(LIB_JSON, encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_library(lib):
    json.dump(lib, open(LIB_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

# ---------- 参考音频转写（ASR） ----------
_whisper = None
def get_whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        _whisper = WhisperModel(
            "small",
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type="float16" if torch.cuda.is_available() else "int8",
        )
    return _whisper

def do_transcribe(ref_audio):
    if not ref_audio:
        return "", "⚠️ 请先上传参考音频。"
    try:
        model = get_whisper()
        segments, _ = model.transcribe(ref_audio, beam_size=1)
        text = "".join(s.text for s in segments).strip()
        return text, "✅ 识别完成，请核对并更正下方文字（文字越准，克隆越像）。"
    except Exception as e:
        return "", "⚠️ 识别失败：" + str(e) + "（可手动填写参考音频说了什么）"

# ---------- 音色库操作 ----------
def do_save_voice(name, ref_audio, ref_text):
    if not name or not name.strip():
        raise gr.Error("请先填写音色名称。")
    if not ref_audio:
        raise gr.Error("请先上传参考音频。")
    name = name.strip()
    lib = load_library()
    ext = os.path.splitext(ref_audio)[1].lower() or ".wav"
    dst = os.path.join(LIB_DIR, "%02d_%d%s" % (len(lib) + 1, int(time.time()), ext))
    shutil.copy(ref_audio, dst)
    lib[name] = {"file": os.path.basename(dst), "prompt_text": (ref_text or "").strip()}
    save_library(lib)
    choices = list(lib.keys())
    return gr.update(choices=choices, value=name), "✅ 已保存音色「%s」到音色库（共 %d 个）。" % (name, len(choices))

def do_delete_voice(lib_voice):
    lib = load_library()
    if lib_voice in lib:
        _f = lib.pop(lib_voice)
        try:
            os.remove(os.path.join(LIB_DIR, _f["file"]))
        except Exception:
            pass
        save_library(lib)
    choices = list(lib.keys())
    return gr.update(choices=choices, value=None), "已删除音色「%s」。" % lib_voice

def preview_preset(preset):
    if not preset:
        return None, "默认音色无需试听，直接合成即可。"
    path = os.path.join(PRESET_DIR, preset + ".wav")
    if not os.path.exists(path):
        return None, "⚠️ 预设音频不存在。"
    data, sr = sf.read(path)
    return (sr, data), "试听预设音色：%s" % PRESET_LABELS.get(preset, preset)

# ---------- 合成 ----------
def synth(source, preset, ref_audio, ref_text, lib_voice, synth_text, synth_lang, speaker_scale,
          seed=0, num_steps=10, guidance_scale=1.2, normalize_text=False):
    prompt_path = None
    prompt_text = None
    info = []
    if source == "音色预设":
        if preset:
            prompt_path = os.path.join(PRESET_DIR, preset + ".wav")
            prompt_text = PRESET_TEXTS.get(preset, "")
            info.append("音色预设：" + PRESET_LABELS.get(preset, preset))
    elif source == "上传参考音频":
        if ref_audio:
            prompt_path = ref_audio
            prompt_text = (ref_text or "").strip() or None
            info.append("音色：上传参考音频")
    else:
        if lib_voice:
            _e = load_library().get(lib_voice)
            if _e:
                prompt_path = os.path.join(LIB_DIR, _e["file"])
                prompt_text = _e.get("prompt_text") or None
                info.append("音色库：" + lib_voice)
    if not synth_text or not synth_text.strip():
        raise gr.Error("请先输入要合成的文字。")
    lang = synth_lang or "auto_detect"
    if seed and int(seed) > 0:
        seed_everything(int(seed))
        info.append("音色种子 %d" % int(seed))
    res = runtime.generate(text=synth_text.strip(), language=lang,
                           prompt_audio_path=prompt_path, prompt_text=prompt_text,
                           speaker_scale=speaker_scale,
                           num_steps=int(num_steps), guidance_scale=float(guidance_scale),
                           normalize_text=bool(normalize_text))
    audio = res["audio"].float().cpu().squeeze().numpy()
    sr = res["sample_rate"]
    dur = round(len(audio) / sr, 2)
    info.append("语言：" + lang)
    info.append("%d 秒 · %d Hz" % (round(dur), sr))
    if prompt_path:
        info.append("音色相似度 %.1f" % speaker_scale)
    else:
        info.append("未用参考音色（模型默认声音）")
    return (sr, audio), " · ".join(info)

# ---------- 音色来源切换：控制各区域显示 ----------
def on_source_change(src):
    if src == "音色预设":
        return (gr.update(visible=True), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False))
    if src == "上传参考音频":
        return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=True), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=True), gr.update(visible=True), gr.update(visible=False),
                gr.update(visible=False))
    # 音色库
    return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=True),
            gr.update(visible=True))

# ---------- 顶部 Banner（标题 + 说明 + 联系链接） ----------
BILIBILI_URL = "https://space.bilibili.com/380877309"
DAOYAKE_URL = "https://www.daoyanke.cn"

_BANNER_HTML = (
    '<div style="text-align:center;padding:20px 14px;background:linear-gradient(135deg,#667eea,#764ba2);border-radius:14px;margin-bottom:14px;">'
    '<h1 style="color:#fff;margin:0 0 6px;font-size:28px;">🎙️ dots.tts 语音合成面板</h1>'
    '<p style="color:#eaeaff;margin:0 0 14px;font-size:15px;">输入文字 → 选音色（预设 / 上传克隆 / 音色库）→ 选语言 → 一键合成<br>支持声音克隆 · 20+ 语言 · 中文方言口音</p>'
    '<p style="margin:0;">'
    '<a href="' + BILIBILI_URL + '" target="_blank" rel="noopener" style="display:inline-block;color:#fff;background:rgba(255,255,255,0.22);padding:5px 16px;border-radius:18px;text-decoration:none;margin:0 6px;font-weight:600;">📺 B站</a>'
    '<a href="' + DAOYAKE_URL + '" target="_blank" rel="noopener" style="display:inline-block;color:#fff;background:rgba(255,255,255,0.22);padding:5px 16px;border-radius:18px;text-decoration:none;margin:0 6px;font-weight:600;">🎬 导演课</a>'
    '</p></div>'
)

# ---------- 界面 ----------
with gr.Blocks(title="dots.tts 语音合成面板") as demo:
    gr.HTML(_BANNER_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## ① 音色设置")
            source = gr.Radio(["音色预设", "上传参考音频", "音色库"], value="音色预设", label="音色来源")
            preset_dd = gr.Dropdown(PRESET_CHOICES, value="", label="音色预设")
            preview_btn = gr.Button("试听预设音色")
            gr.Markdown("💡 **调情绪/语气**：情绪来自参考音频的韵律——选「情绪参考」预设，或上传带目标情绪的人声（3-10 秒）；再用下方「音色种子」换韵律。")
            preview_audio = gr.Audio(label="预设试听")
            ref_audio = gr.Audio(label="参考音频（3-10 秒清晰人声）", type="filepath", visible=False)
            transcribe_btn = gr.Button("识别转写", visible=False)
            ref_text = gr.Textbox(label="参考音频文字（自动识别，可手动更正）", lines=3, visible=False,
                                  placeholder="上传音频后点「识别转写」自动填写；也可直接手填。文字越准，克隆越像。")
            voice_name = gr.Textbox(label="音色名称（保存到音色库）", visible=False, placeholder="例如：我的声音")
            save_btn = gr.Button("保存到音色库", visible=False)
            lib_dd = gr.Dropdown(choices=list(load_library().keys()), value=None,
                                 label="音色库（已保存的音色）", visible=False)
            delete_btn = gr.Button("删除选中音色", visible=False)
            voice_status = gr.Textbox(label="提示", interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("## ② 合成")
            synth_text = gr.Textbox(label="要合成的文字", lines=4, value="你好，欢迎使用 dots.tts 语音合成面板。")
            synth_lang = gr.Dropdown(LANG_CHOICES, value="ZH", label="语言")
            speaker_scale = gr.Slider(minimum=0.5, maximum=3.0, value=1.5, step=0.1,
                                      label="音色相似度（使用参考音色时生效，越高越像）")
            with gr.Accordion("⚙️ 高级设置（可选）", open=False):
                seed = gr.Slider(minimum=0, maximum=9999, value=0, step=1,
                                 label="音色种子（0=随机；固定数字=每次生成同一个声音）")
                num_steps = gr.Slider(minimum=10, maximum=32, value=10, step=1,
                                      label="生成质量·采样步数（越大越细腻，但更慢）")
                guidance_scale = gr.Slider(minimum=0.5, maximum=3.0, value=1.2, step=0.1,
                                           label="引导强度（越大越贴合文字与参考音色）")
                normalize_text = gr.Checkbox(value=False, label="文本规范化（数字/符号自动转口语读法）")
            synth_btn = gr.Button("开始合成", variant="primary")
            result_audio = gr.Audio(label="合成结果")
            result_info = gr.Textbox(label="结果信息", interactive=False)

    _voice_components = [preset_dd, preview_btn, preview_audio, ref_audio, transcribe_btn,
                         ref_text, voice_name, save_btn, lib_dd, delete_btn]
    source.change(on_source_change, source, _voice_components)
    preview_btn.click(preview_preset, preset_dd, [preview_audio, voice_status])
    transcribe_btn.click(do_transcribe, ref_audio, [ref_text, voice_status])
    save_btn.click(do_save_voice, [voice_name, ref_audio, ref_text], [lib_dd, voice_status])
    delete_btn.click(do_delete_voice, lib_dd, [lib_dd, voice_status])
    synth_btn.click(synth, [source, preset_dd, ref_audio, ref_text, lib_dd, synth_text, synth_lang,
                            speaker_scale, seed, num_steps, guidance_scale, normalize_text],
                    [result_audio, result_info])

print("启动 Gradio 面板（share=True，正在建立公网隧道）...", flush=True)
demo.launch(share=True, debug=False)
"""
open("/content/panel.py", "w", encoding="utf-8").write(panel_code)
print("✅ 面板代码已写入 /content/panel.py", flush=True)

drive_hub = os.path.join(CACHE, "hub") if DRIVE_OK else None
local_hub = os.path.join(LOCAL_HF, "hub")

if drive_hub and os.path.isdir(drive_hub):
    if not os.path.isdir(local_hub):
        print("📦 复制模型缓存到本地 SSD（含 blobs 软链，约 1-3 分钟，之后加载飞快）...", flush=True)
        os.makedirs(LOCAL_HF, exist_ok=True)
        subprocess.run(["cp", "-a", drive_hub, local_hub], check=True)
        print("✅ 模型已就位本地 SSD", flush=True)
    else:
        print("✅ 模型已在本地 SSD（本次会话已复制过，跳过）", flush=True)
    HF_HOME_USE = LOCAL_HF
else:
    # 首次运行：还没有 Drive 缓存，模型将直接下载到 Drive
    HF_HOME_USE = CACHE if DRIVE_OK else None
    print("ℹ️ 首次运行：模型将下载到", HF_HOME_USE or "默认缓存", flush=True)


## 第 4 步：启动面板 + 等待公网地址

启动前会**先杀掉旧面板进程**（避免抢 GPU 导致加载失败），然后智能等待地址：最多等 **20 分钟**，期间若进程崩了会立即停下并打印日志末尾，方便定位。


In [ ]:
# ---- 第 4 步：启动面板 + 智能等待公网地址 ----
import subprocess, os, time, re

# 杀掉可能残留的旧面板进程（避免抢 GPU 导致加载失败）
subprocess.run("pkill -f panel.py || true", shell=True)
time.sleep(2)

env = dict(os.environ)
if 'HF_HOME_USE' in dir() and HF_HOME_USE:
    env["HF_HOME"] = HF_HOME_USE
elif DRIVE_OK and os.path.isdir(CACHE):
    env["HF_HOME"] = CACHE

proc = subprocess.Popen([PY, "-u", "/content/panel.py"],
                        stdout=open("panel.log", "w"),
                        stderr=subprocess.STDOUT, env=env)

URL_RE = re.compile(r"https://[a-z0-9.-]+\.gradio\.live")
url = None
t0 = time.time()
i = 0
while time.time() - t0 < 1200:          # 最多等 20 分钟
    if proc.poll() is not None:          # 进程已退出
        break
    if os.path.exists("panel.log"):
        text = open("panel.log", encoding="utf-8", errors="ignore").read()
        m = URL_RE.search(text)
        if m:
            url = m.group(0)
            break
    i += 1
    if i % 15 == 0:
        print("  ... 已等 %d 秒" % (i * 2), flush=True)
    time.sleep(2)

if url:
    open("panel_url.txt", "w").write(url)
    print("🌐 面板公网地址：", url, flush=True)
    print("   用浏览器打开这个地址（保持梯子开启）。", flush=True)
else:
    print("⚠️ 未获取到地址。", flush=True)
    if proc.poll() is not None:
        print("面板进程已退出，退出码：", proc.returncode, flush=True)
    if os.path.exists("panel.log"):
        tail = open("panel.log", encoding="utf-8", errors="ignore").read()[-3000:]
        print("日志末尾：", flush=True)
        print(re.sub(r"\x1b\[[0-9;]*m", "", tail), flush=True)
    else:
        print("无日志", flush=True)


## 🚀 一键启动（断连后 / 以后每次只跑这一格）

这一格 = 第 1 + 2 + 3 + 4 步的合体。**首次安装完成后，以后每次（含断连重开）只跑这一格就行**，约 2-4 分钟出地址。


In [ ]:
# ---- 第 1 步：挂载 Google Drive + 定义路径 ----
import os, subprocess, time, re, shutil, sys

CACHE = "/content/drive/MyDrive/dots_cache"        # Drive 持久化目录（环境 + 模型 + 音色库）
PY = "/content/py311/bin/python"                    # 本地 Python 环境
ENV_TARBALL = os.path.join(CACHE, "py311.tar.gz")   # 环境备份包（首次装好后存到 Drive）
LOCAL_HF = "/content/dots_hf_cache"                 # 本地 SSD 模型缓存（加载快）
PANEL_PY = "/content/panel.py"

try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    DRIVE_OK = True
    print("✅ Drive 已挂载，持久化目录：", CACHE, flush=True)
except Exception as e:
    DRIVE_OK = False
    print("⚠️ Drive 未挂载（断连后需重装环境 + 重下模型）：", e, flush=True)


# ---- 第 2 步：准备环境（首次安装并缓存到 Drive，之后秒恢复；缺依赖会自动补齐）----
import os, subprocess

_REQ = ["torch", "gradio", "dots_tts", "faster_whisper", "soundfile", "huggingface_hub"]

def _py_runs(p):
    try:
        return subprocess.run([p, "--version"], capture_output=True, text=True, timeout=60).returncode == 0
    except Exception:
        return False

def _deps_ok(p):
    _code = "import importlib.util as u, sys; sys.exit(0 if all(u.find_spec(m) for m in %r) else 1)" % (_REQ,)
    try:
        return subprocess.run([p, "-c", _code], capture_output=True, text=True, timeout=120).returncode == 0
    except Exception:
        return False

def _install_deps():
    subprocess.run("pip install -q uv", shell=True, check=True)
    UV = "uv pip install --python /content/py311/bin/python"
    subprocess.run(UV + " torch==2.11.0 torchaudio==2.11.0", shell=True, check=True)
    subprocess.run(UV + " dots.tts huggingface_hub soundfile 'gradio>=6.17,<7' faster-whisper", shell=True, check=True)

# 1) 确保 python 解释器存在（恢复 / 新建）
if not _py_runs(PY):
    if DRIVE_OK and os.path.exists(ENV_TARBALL):
        print("🔄 从 Drive 恢复环境（约 1-2 分钟，免重装）...", flush=True)
        subprocess.run(["tar", "xzf", ENV_TARBALL, "-C", "/"], check=True)
    if not _py_runs(PY):
        print("🔄 首次安装环境（约 3-5 分钟）...", flush=True)
        subprocess.run("python3 -m venv /content/py311", shell=True, check=True)

# 2) 检查依赖是否齐全，缺哪个补哪个（uv 幂等，已装的秒过）
if not _deps_ok(PY):
    print("🔄 检测到依赖缺失，正在补齐（已装的会自动跳过）...", flush=True)
    _install_deps()

if not _deps_ok(PY):
    _r = subprocess.run([PY, "-c", "import gradio"], capture_output=True, text=True)
    print("❌ 依赖仍缺失：", _r.stderr[-800:], flush=True)
    raise SystemExit("依赖安装失败，请检查上方报错后重跑本格")

# 3) 环境齐了，首次打包缓存到 Drive（之后断连免重装）
if DRIVE_OK and not os.path.exists(ENV_TARBALL):
    print("📦 缓存环境到 Drive（首次稍慢，约 2-4 分钟）...", flush=True)
    subprocess.run(["tar", "czf", ENV_TARBALL, "-C", "/", "content/py311"], check=True)
    print("✅ 环境已缓存：", ENV_TARBALL, flush=True)

print("✅ 环境就绪（含 gradio / torch / dots.tts / faster-whisper）", flush=True)


# ---- 第 3 步：准备模型（复制到本地 SSD，加载快）+ 写面板代码 ----
import os, subprocess

panel_code = r"""import os, json, shutil, time
import gradio as gr
import soundfile as sf
import torch
from dots_tts.runtime import DotsTtsRuntime
from dots_tts.utils.util import seed_everything

# ---------- 加载模型 ----------
print("HF_HOME =", os.environ.get("HF_HOME"), flush=True)
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
PRECISION = "bfloat16" if cap[0] >= 8 else "float16"
print("加载模型...", flush=True)
runtime = DotsTtsRuntime.from_pretrained("dots-studio/dots.tts-soar", precision=PRECISION, optimize=False)
print("模型加载完成", flush=True)

# ---------- 目录 ----------
CACHE = os.environ.get("HF_HOME") or ("/content/drive/MyDrive/dots_cache" if os.path.isdir("/content/drive/MyDrive") else None)
LIB_DIR = os.path.join(CACHE, "voice_library") if CACHE else "/content/voice_library"
PRESET_DIR = "/content/presets"
os.makedirs(LIB_DIR, exist_ok=True)
os.makedirs(PRESET_DIR, exist_ok=True)
LIB_JSON = os.path.join(LIB_DIR, "voices.json")

# ---------- 内置音色预设（运行时从 GitHub 仓库下载参考音频，避免内嵌 base64 拖慢代码页） ----------
# 每项：(key, 标签, 文件名, 参考文本)。参考文本必须与音频实际内容一致。
PRESET_DEFS = [
    # 中文
    ("tingting", "婷婷（温柔女声）", "tingting.wav", "大家好，我是你的专属语音助手。今天天气很不错，我们一起来聊一聊最近发生的趣事吧。"),
    ("eddy", "埃迪（沉稳男声）", "eddy.wav", "各位听众朋友，大家好。欢迎收听今天的节目，希望你能喜欢我的声音，也祝你度过愉快的一天。"),
    ("meijia", "美佳（甜美女声）", "meijia.wav", "你好呀，很高兴认识你。今天想跟你分享一些有趣的事情，希望你听了以后会开心一点。"),
    ("sandy", "桑迪（知性女声）", "sandy.wav", "大家好，我是你的语音助手。无论是工作还是生活，我都愿意随时为你提供帮助和建议。"),
    # 英语
    ("samantha", "Samantha（美式女声）", "samantha.wav", "Hello, I'm your friendly voice assistant. It's a pleasure to meet you, and I'm here to help with whatever you need today."),
    ("daniel", "Daniel（英式男声）", "daniel.wav", "Good day to you, and welcome. I hope you find my voice clear, natural, and pleasant to listen to."),
    ("karen", "Karen（澳洲女声）", "karen.wav", "G'day, I'm your voice assistant. Let's get started and make the most of today, together."),
    ("moira", "Moira（爱尔兰女声）", "moira.wav", "Hello there, lovely to meet you. I'll be guiding you through today, so just relax and enjoy the conversation."),
    # 其他语言
    ("kyoko", "Kyoko（日语女声）", "kyoko.wav", "こんにちは、あなたの音声アシスタントです。今日もよろしくお願いします。"),
    ("yuna", "Yuna（韩语女声）", "yuna.wav", "안녕하세요, 저는 당신의 음성 비서입니다. 오늘도 좋은 하루 보내세요."),
    ("thomas", "Thomas（法语男声）", "thomas.wav", "Bonjour, je suis votre assistant vocal. C'est un plaisir de vous accompagner aujourd'hui."),
    ("anna", "Anna（德语女声）", "anna.wav", "Hallo, ich bin deine Sprachassistentin. Schön, dich heute begleiten zu dürfen."),
    # 情绪参考（英文，用来把情绪/语气转移到合成结果；合成中文会带英文口音，适合做情绪参考）
    ("goodnews", "开心·欢快（情绪参考）", "goodnews.wav", "Great news! Everything went perfectly today!"),
    ("badnews", "悲伤·低沉（情绪参考）", "badnews.wav", "I'm afraid I have some difficult news to share with you."),
    ("whisper", "悄悄话·耳语（情绪参考）", "whisper.wav", "Psst, come a little closer. I have a secret to tell you, but just between us, quietly."),
]
PRESET_BASE = "https://raw.githubusercontent.com/Evan78s/dots-tts-panel/main/presets"

PRESET_LABELS = {}
PRESET_TEXTS = {}
for _key, _label, _file, _text in PRESET_DEFS:
    PRESET_LABELS[_key] = _label
    PRESET_TEXTS[_key] = _text

def _ensure_presets():
    import urllib.request
    for _key, _label, _file, _text in PRESET_DEFS:
        _path = os.path.join(PRESET_DIR, _file)
        if os.path.exists(_path) and os.path.getsize(_path) > 1000:
            continue
        try:
            urllib.request.urlretrieve(PRESET_BASE + "/" + _file, _path)
            print("下载预设音色：%s" % _label, flush=True)
        except Exception as _e:
            print("⚠️ 预设音色「%s」下载失败（仍可上传参考音频使用）：%s" % (_label, _e), flush=True)

_ensure_presets()
print("内置音色预设：", list(PRESET_LABELS.values()), flush=True)

PRESET_CHOICES = [("默认音色（不克隆）", "")] + [(lbl, key) for key, lbl in PRESET_LABELS.items()]

# ---------- 语言（全部中文显示） ----------
LANG_CHOICES = [
    ("自动检测", "auto_detect"),
    ("中文（普通话）", "ZH"),
    ("英语", "EN"),
    ("粤语", "Cantonese"),
    ("日语", "JA"),
    ("韩语", "KO"),
    ("法语", "FR"),
    ("德语", "DE"),
    ("西班牙语", "ES"),
    ("俄语", "RU"),
    ("阿拉伯语", "AR"),
    ("印地语", "HI"),
    ("葡萄牙语", "PT"),
    ("意大利语", "IT"),
    ("泰语", "TH"),
    ("越南语", "VI"),
    ("印尼语", "ID"),
    ("捷克语", "CS"),
    ("荷兰语", "NL"),
    ("芬兰语", "FI"),
    ("希腊语", "EL"),
    ("波兰语", "PL"),
    ("罗马尼亚语", "RO"),
    ("土耳其语", "TR"),
    ("乌克兰语", "UK"),
    ("口音：北京官话", "口音:北京官话"),
    ("口音：东北话", "口音:东北话"),
    ("口音：四川话", "口音:四川话"),
    ("口音：闽南话", "口音:闽南话"),
    ("口音：吴语", "口音:吴语"),
]

# ---------- 音色库（持久化到 Drive） ----------
def load_library():
    if os.path.exists(LIB_JSON):
        try:
            return json.load(open(LIB_JSON, encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_library(lib):
    json.dump(lib, open(LIB_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

# ---------- 参考音频转写（ASR） ----------
_whisper = None
def get_whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        _whisper = WhisperModel(
            "small",
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type="float16" if torch.cuda.is_available() else "int8",
        )
    return _whisper

def do_transcribe(ref_audio):
    if not ref_audio:
        return "", "⚠️ 请先上传参考音频。"
    try:
        model = get_whisper()
        segments, _ = model.transcribe(ref_audio, beam_size=1)
        text = "".join(s.text for s in segments).strip()
        return text, "✅ 识别完成，请核对并更正下方文字（文字越准，克隆越像）。"
    except Exception as e:
        return "", "⚠️ 识别失败：" + str(e) + "（可手动填写参考音频说了什么）"

# ---------- 音色库操作 ----------
def do_save_voice(name, ref_audio, ref_text):
    if not name or not name.strip():
        raise gr.Error("请先填写音色名称。")
    if not ref_audio:
        raise gr.Error("请先上传参考音频。")
    name = name.strip()
    lib = load_library()
    ext = os.path.splitext(ref_audio)[1].lower() or ".wav"
    dst = os.path.join(LIB_DIR, "%02d_%d%s" % (len(lib) + 1, int(time.time()), ext))
    shutil.copy(ref_audio, dst)
    lib[name] = {"file": os.path.basename(dst), "prompt_text": (ref_text or "").strip()}
    save_library(lib)
    choices = list(lib.keys())
    return gr.update(choices=choices, value=name), "✅ 已保存音色「%s」到音色库（共 %d 个）。" % (name, len(choices))

def do_delete_voice(lib_voice):
    lib = load_library()
    if lib_voice in lib:
        _f = lib.pop(lib_voice)
        try:
            os.remove(os.path.join(LIB_DIR, _f["file"]))
        except Exception:
            pass
        save_library(lib)
    choices = list(lib.keys())
    return gr.update(choices=choices, value=None), "已删除音色「%s」。" % lib_voice

def preview_preset(preset):
    if not preset:
        return None, "默认音色无需试听，直接合成即可。"
    path = os.path.join(PRESET_DIR, preset + ".wav")
    if not os.path.exists(path):
        return None, "⚠️ 预设音频不存在。"
    data, sr = sf.read(path)
    return (sr, data), "试听预设音色：%s" % PRESET_LABELS.get(preset, preset)

# ---------- 合成 ----------
def synth(source, preset, ref_audio, ref_text, lib_voice, synth_text, synth_lang, speaker_scale,
          seed=0, num_steps=10, guidance_scale=1.2, normalize_text=False):
    prompt_path = None
    prompt_text = None
    info = []
    if source == "音色预设":
        if preset:
            prompt_path = os.path.join(PRESET_DIR, preset + ".wav")
            prompt_text = PRESET_TEXTS.get(preset, "")
            info.append("音色预设：" + PRESET_LABELS.get(preset, preset))
    elif source == "上传参考音频":
        if ref_audio:
            prompt_path = ref_audio
            prompt_text = (ref_text or "").strip() or None
            info.append("音色：上传参考音频")
    else:
        if lib_voice:
            _e = load_library().get(lib_voice)
            if _e:
                prompt_path = os.path.join(LIB_DIR, _e["file"])
                prompt_text = _e.get("prompt_text") or None
                info.append("音色库：" + lib_voice)
    if not synth_text or not synth_text.strip():
        raise gr.Error("请先输入要合成的文字。")
    lang = synth_lang or "auto_detect"
    if seed and int(seed) > 0:
        seed_everything(int(seed))
        info.append("音色种子 %d" % int(seed))
    res = runtime.generate(text=synth_text.strip(), language=lang,
                           prompt_audio_path=prompt_path, prompt_text=prompt_text,
                           speaker_scale=speaker_scale,
                           num_steps=int(num_steps), guidance_scale=float(guidance_scale),
                           normalize_text=bool(normalize_text))
    audio = res["audio"].float().cpu().squeeze().numpy()
    sr = res["sample_rate"]
    dur = round(len(audio) / sr, 2)
    info.append("语言：" + lang)
    info.append("%d 秒 · %d Hz" % (round(dur), sr))
    if prompt_path:
        info.append("音色相似度 %.1f" % speaker_scale)
    else:
        info.append("未用参考音色（模型默认声音）")
    return (sr, audio), " · ".join(info)

# ---------- 音色来源切换：控制各区域显示 ----------
def on_source_change(src):
    if src == "音色预设":
        return (gr.update(visible=True), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False))
    if src == "上传参考音频":
        return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=True), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=True), gr.update(visible=True), gr.update(visible=False),
                gr.update(visible=False))
    # 音色库
    return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=True),
            gr.update(visible=True))

# ---------- 顶部 Banner（标题 + 说明 + 联系链接） ----------
BILIBILI_URL = "https://space.bilibili.com/380877309"
DAOYAKE_URL = "https://www.daoyanke.cn"

_BANNER_HTML = (
    '<div style="text-align:center;padding:20px 14px;background:linear-gradient(135deg,#667eea,#764ba2);border-radius:14px;margin-bottom:14px;">'
    '<h1 style="color:#fff;margin:0 0 6px;font-size:28px;">🎙️ dots.tts 语音合成面板</h1>'
    '<p style="color:#eaeaff;margin:0 0 14px;font-size:15px;">输入文字 → 选音色（预设 / 上传克隆 / 音色库）→ 选语言 → 一键合成<br>支持声音克隆 · 20+ 语言 · 中文方言口音</p>'
    '<p style="margin:0;">'
    '<a href="' + BILIBILI_URL + '" target="_blank" rel="noopener" style="display:inline-block;color:#fff;background:rgba(255,255,255,0.22);padding:5px 16px;border-radius:18px;text-decoration:none;margin:0 6px;font-weight:600;">📺 B站</a>'
    '<a href="' + DAOYAKE_URL + '" target="_blank" rel="noopener" style="display:inline-block;color:#fff;background:rgba(255,255,255,0.22);padding:5px 16px;border-radius:18px;text-decoration:none;margin:0 6px;font-weight:600;">🎬 导演课</a>'
    '</p></div>'
)

# ---------- 界面 ----------
with gr.Blocks(title="dots.tts 语音合成面板") as demo:
    gr.HTML(_BANNER_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## ① 音色设置")
            source = gr.Radio(["音色预设", "上传参考音频", "音色库"], value="音色预设", label="音色来源")
            preset_dd = gr.Dropdown(PRESET_CHOICES, value="", label="音色预设")
            preview_btn = gr.Button("试听预设音色")
            gr.Markdown("💡 **调情绪/语气**：情绪来自参考音频的韵律——选「情绪参考」预设，或上传带目标情绪的人声（3-10 秒）；再用下方「音色种子」换韵律。")
            preview_audio = gr.Audio(label="预设试听")
            ref_audio = gr.Audio(label="参考音频（3-10 秒清晰人声）", type="filepath", visible=False)
            transcribe_btn = gr.Button("识别转写", visible=False)
            ref_text = gr.Textbox(label="参考音频文字（自动识别，可手动更正）", lines=3, visible=False,
                                  placeholder="上传音频后点「识别转写」自动填写；也可直接手填。文字越准，克隆越像。")
            voice_name = gr.Textbox(label="音色名称（保存到音色库）", visible=False, placeholder="例如：我的声音")
            save_btn = gr.Button("保存到音色库", visible=False)
            lib_dd = gr.Dropdown(choices=list(load_library().keys()), value=None,
                                 label="音色库（已保存的音色）", visible=False)
            delete_btn = gr.Button("删除选中音色", visible=False)
            voice_status = gr.Textbox(label="提示", interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("## ② 合成")
            synth_text = gr.Textbox(label="要合成的文字", lines=4, value="你好，欢迎使用 dots.tts 语音合成面板。")
            synth_lang = gr.Dropdown(LANG_CHOICES, value="ZH", label="语言")
            speaker_scale = gr.Slider(minimum=0.5, maximum=3.0, value=1.5, step=0.1,
                                      label="音色相似度（使用参考音色时生效，越高越像）")
            with gr.Accordion("⚙️ 高级设置（可选）", open=False):
                seed = gr.Slider(minimum=0, maximum=9999, value=0, step=1,
                                 label="音色种子（0=随机；固定数字=每次生成同一个声音）")
                num_steps = gr.Slider(minimum=10, maximum=32, value=10, step=1,
                                      label="生成质量·采样步数（越大越细腻，但更慢）")
                guidance_scale = gr.Slider(minimum=0.5, maximum=3.0, value=1.2, step=0.1,
                                           label="引导强度（越大越贴合文字与参考音色）")
                normalize_text = gr.Checkbox(value=False, label="文本规范化（数字/符号自动转口语读法）")
            synth_btn = gr.Button("开始合成", variant="primary")
            result_audio = gr.Audio(label="合成结果")
            result_info = gr.Textbox(label="结果信息", interactive=False)

    _voice_components = [preset_dd, preview_btn, preview_audio, ref_audio, transcribe_btn,
                         ref_text, voice_name, save_btn, lib_dd, delete_btn]
    source.change(on_source_change, source, _voice_components)
    preview_btn.click(preview_preset, preset_dd, [preview_audio, voice_status])
    transcribe_btn.click(do_transcribe, ref_audio, [ref_text, voice_status])
    save_btn.click(do_save_voice, [voice_name, ref_audio, ref_text], [lib_dd, voice_status])
    delete_btn.click(do_delete_voice, lib_dd, [lib_dd, voice_status])
    synth_btn.click(synth, [source, preset_dd, ref_audio, ref_text, lib_dd, synth_text, synth_lang,
                            speaker_scale, seed, num_steps, guidance_scale, normalize_text],
                    [result_audio, result_info])

print("启动 Gradio 面板（share=True，正在建立公网隧道）...", flush=True)
demo.launch(share=True, debug=False)
"""
open("/content/panel.py", "w", encoding="utf-8").write(panel_code)
print("✅ 面板代码已写入 /content/panel.py", flush=True)

drive_hub = os.path.join(CACHE, "hub") if DRIVE_OK else None
local_hub = os.path.join(LOCAL_HF, "hub")

if drive_hub and os.path.isdir(drive_hub):
    if not os.path.isdir(local_hub):
        print("📦 复制模型缓存到本地 SSD（含 blobs 软链，约 1-3 分钟，之后加载飞快）...", flush=True)
        os.makedirs(LOCAL_HF, exist_ok=True)
        subprocess.run(["cp", "-a", drive_hub, local_hub], check=True)
        print("✅ 模型已就位本地 SSD", flush=True)
    else:
        print("✅ 模型已在本地 SSD（本次会话已复制过，跳过）", flush=True)
    HF_HOME_USE = LOCAL_HF
else:
    # 首次运行：还没有 Drive 缓存，模型将直接下载到 Drive
    HF_HOME_USE = CACHE if DRIVE_OK else None
    print("ℹ️ 首次运行：模型将下载到", HF_HOME_USE or "默认缓存", flush=True)


# ---- 第 4 步：启动面板 + 智能等待公网地址 ----
import subprocess, os, time, re

# 杀掉可能残留的旧面板进程（避免抢 GPU 导致加载失败）
subprocess.run("pkill -f panel.py || true", shell=True)
time.sleep(2)

env = dict(os.environ)
if 'HF_HOME_USE' in dir() and HF_HOME_USE:
    env["HF_HOME"] = HF_HOME_USE
elif DRIVE_OK and os.path.isdir(CACHE):
    env["HF_HOME"] = CACHE

proc = subprocess.Popen([PY, "-u", "/content/panel.py"],
                        stdout=open("panel.log", "w"),
                        stderr=subprocess.STDOUT, env=env)

URL_RE = re.compile(r"https://[a-z0-9.-]+\.gradio\.live")
url = None
t0 = time.time()
i = 0
while time.time() - t0 < 1200:          # 最多等 20 分钟
    if proc.poll() is not None:          # 进程已退出
        break
    if os.path.exists("panel.log"):
        text = open("panel.log", encoding="utf-8", errors="ignore").read()
        m = URL_RE.search(text)
        if m:
            url = m.group(0)
            break
    i += 1
    if i % 15 == 0:
        print("  ... 已等 %d 秒" % (i * 2), flush=True)
    time.sleep(2)

if url:
    open("panel_url.txt", "w").write(url)
    print("🌐 面板公网地址：", url, flush=True)
    print("   用浏览器打开这个地址（保持梯子开启）。", flush=True)
else:
    print("⚠️ 未获取到地址。", flush=True)
    if proc.poll() is not None:
        print("面板进程已退出，退出码：", proc.returncode, flush=True)
    if os.path.exists("panel.log"):
        tail = open("panel.log", encoding="utf-8", errors="ignore").read()[-3000:]
        print("日志末尾：", flush=True)
        print(re.sub(r"\x1b\[[0-9;]*m", "", tail), flush=True)
    else:
        print("无日志", flush=True)


## ❓ 常见问题

| 问题 | 解决 |
|---|---|
| 首次很慢 | 正常：装环境 + 下 5GB 模型，约 5-8 分钟 |
| 断连后还要重装吗 | 不用了。环境已打包到 Drive，重开跑「一键启动」约 2-4 分钟 |
| 面板地址打不开 | 大陆用户需挂梯子（跟访问 Colab 同一个）；或换「全局模式」 |
| 等了很久没地址 | 最多等 20 分钟；若进程报错会打印日志末尾，照着修 |
| 转写报错 / 组件缺失 | 环境已内置 faster-whisper；仍报错可删 Drive 的 `py311.tar.gz` 重装一次 |
| 音色下次不见了 | 需挂载 Drive（音色库存 `dots_cache/voice_library/`） |
| 想彻底重装 | 删除 Drive 的 `dots_cache/py311.tar.gz`，再跑「一键启动」会自动重装 |
